# BPI Repossessed Cars (PDF)

In [1]:
!pip install PyMuPDF pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 9.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 24.1 MB/s eta 0:00:00


In [2]:
import fitz
import re
import pandas as pd

## Read PDF File

In [33]:
pdf_document = '/content/drive/MyDrive/Webscraping Datasets (BPI)/cagayan_de_oro.pdf'
pdf = fitz.open(pdf_document)

## Extract Text

In [34]:
text = ""
for page_num in range(len(pdf)):
    page = pdf.load_page(page_num)
    text += page.get_text()
text

"REG BIDDING\nBID NOW\nPRICE\nPRICE\n2021 SUZUKI ERTIGA M/T 1.5 GL   \n29,155\nJAK7175\nGRAY\n532,707\n585,978\n2020 NISSAN  URVAN M/T 2.5 0  (D)  [ Broken front \nwindshield ]\n349,273\nJAD9285\nWHITE\n478,035\n525,838\n2019 MITSUBISHI STRADA M/T 2.4 GLS 4X2 (D)                                               \n[ UNENCUMBERED CR AND UNANNOTATED PNCM  ]\n5,914\nKAD5720\nSILVER\n558,390\n614,229\n2018 MITSUBISHI MONTERO  A/T 2.4 GLS 4X2 (D)                                  \n[ UNENCUMBERED CR AND UNANNOTATED PNCM, missing \ntail/stoplight ]\n41,658\nNO PLATE\nBLUE\n789,889\n868,878\n2019 NISSAN  NAVARA A/T 2.5 EL 4X2 (D)  [ UNABLE TO \nOPEN HOOD  ]\n83,689\nJAH8296\nBROWN\n564,340\n620,774\n2022 MG MG5 M/T 1.5  [ UNENCUMBERED CR AND \nUNANNOTATED PNCM  ]\n13,967\nLAK5451\nWHITE\n315,140\n346,654\n2020 TOYOTA  VIOS M/T 1.3 J   [ BROKEN LEFT TAILIGHT \n, POOR TRUNK LID CONDITION. UNENCUMBERED CR  ]\n145,534\nGAO1646\nWHITE\n294,724\n324,197\n2018 MITSUBISHI MIRAGE G4 A/T 1.2 GLX   [ BROKEN 

In [37]:
lines = text.split('\n')
lines[4:]

['2021 SUZUKI ERTIGA M/T 1.5 GL   ',
 '29,155',
 'JAK7175',
 'GRAY',
 '532,707',
 '585,978',
 '2020 NISSAN  URVAN M/T 2.5 0  (D)  [ Broken front ',
 'windshield ]',
 '349,273',
 'JAD9285',
 'WHITE',
 '478,035',
 '525,838',
 '2019 MITSUBISHI STRADA M/T 2.4 GLS 4X2 (D)                                               ',
 '[ UNENCUMBERED CR AND UNANNOTATED PNCM  ]',
 '5,914',
 'KAD5720',
 'SILVER',
 '558,390',
 '614,229',
 '2018 MITSUBISHI MONTERO  A/T 2.4 GLS 4X2 (D)                                  ',
 '[ UNENCUMBERED CR AND UNANNOTATED PNCM, missing ',
 'tail/stoplight ]',
 '41,658',
 'NO PLATE',
 'BLUE',
 '789,889',
 '868,878',
 '2019 NISSAN  NAVARA A/T 2.5 EL 4X2 (D)  [ UNABLE TO ',
 'OPEN HOOD  ]',
 '83,689',
 'JAH8296',
 'BROWN',
 '564,340',
 '620,774',
 '2022 MG MG5 M/T 1.5  [ UNENCUMBERED CR AND ',
 'UNANNOTATED PNCM  ]',
 '13,967',
 'LAK5451',
 'WHITE',
 '315,140',
 '346,654',
 '2020 TOYOTA  VIOS M/T 1.3 J   [ BROKEN LEFT TAILIGHT ',
 ', POOR TRUNK LID CONDITION. UNENCUMBERED CR  ]

## Extract Data

In [40]:
def extract_data(lines):
    entry = ' '.join(lines)
    pattern = re.compile(
        r'''(\d{4})  # Year
        \s+(.*)  # Unit Description
        \s+(NO\sMILEAGE|unverified|-|N/A|(?:\d{1,3}(?:,\d{3})*)) # Kilometrage
        \s+([\w/\d]+(?:\s*/\s*[\w/\d]+)?) # PN/CS
        \s+([\w\s/]+) # Color
        \s+((?:\d{1,3}(?:,\d{3})*)) # Reg Bidding Price
        \s+((?:\d{1,3}(?:,\d{3})*)) # Bid Now Price''',
        re.VERBOSE
    )
    match = pattern.search(entry)
    if match:
        return match.groups()
    return None

data = []
temp_lines = []

for line in lines:
    # Check if the line starts with a year (assuming it starts a new entry)
    if re.match(r"\d{4}", line):
        if temp_lines:
            parsed_data = extract_data(temp_lines)
            if parsed_data:
                data.append(parsed_data)
            temp_lines = []

    # Append the current line to temp_lines
    temp_lines.append(line)

# Process the last accumulated lines
if temp_lines:
    parsed_data = extract_data(temp_lines)
    if (parsed_data):
        data.append(parsed_data)

# Define the column names
columns = ["year", "unit_description", "kilometrage", "pn/cs", "color", "reg_bidding_price", "bid_now_price"]

# Create a DataFrame from the extracted data
df = pd.DataFrame(data, columns=columns)

#Add Transmission

for index, row in df.iterrows():
  if 'M/T' in row['unit_description']:
    df.at[index, 'transmission'] = 'M/T'
  elif 'A/T' in row['unit_description']:
    df.at[index, 'transmission'] = 'A/T'

# Display the DataFrame
df

,year,unit_description,kilometrage,pn/cs,color,reg_bidding_price,bid_now_price,transmission
0,2021,SUZUKI ERTIGA M/T 1.5 GL,"29,155",JAK7175,GRAY,"532,707","585,978",M/T
1,2020,NISSAN URVAN M/T 2.5 0 (D) [ Broken front ...,"349,273",JAD9285,WHITE,"478,035","525,838",M/T
2,2019,MITSUBISHI STRADA M/T 2.4 GLS 4X2 (D) ...,"5,914",KAD5720,SILVER,"558,390","614,229",M/T
3,2018,MITSUBISHI MONTERO A/T 2.4 GLS 4X2 (D) ...,"41,658",NO,PLATE BLUE,"789,889","868,878",A/T
4,2019,NISSAN NAVARA A/T 2.5 EL 4X2 (D) [ UNABLE TO...,"83,689",JAH8296,BROWN,"564,340","620,774",A/T
5,2022,MG MG5 M/T 1.5 [ UNENCUMBERED CR AND UNANNOT...,"13,967",LAK5451,WHITE,"315,140","346,654",M/T
6,2020,TOYOTA VIOS M/T 1.3 J [ BROKEN LEFT TAILIGH...,"145,534",GAO1646,WHITE,"294,724","324,197",M/T
7,2018,MITSUBISHI MIRAGE G4 A/T 1.2 GLX [ BROKEN L...,"55,730",KAB8123,GRAY,"261,064","287,171",A/T
8,2020,CHEVROLET COLORADO A/T 2.8 TRAIL BOSS 4X2 (...,"117,640",KAF8625,BROWN,"583,977","642,374",A/T
9,2023,KIA SOLUTO A/T 1.4 LX [ UNENCUMBERED CR ],"9,729",XOT739,BEIGE,"420,210","462,231",A/T


In [39]:
test = ['2020 NISSAN  NAVARA M/T 2.5 EL 4X2 (D)  [ No battery ]',
 'NO MILEAGE ',
 'JAD5115',
 'BROWN',
 '588,640',
 '647,504']

test1 = ['2022 Changan CS35 A/T 1.4L Plus 7DCT Luxe   [ No seat ',
 'covers. LTO Cebu City-registered. Cancellation of mortgage is ',
 'at RD-Negros Oriental. ]',
 '15,378',
 'N1P916',
 'White',
 '535,990',
 '589,589']


data = []
temp_lines = []

for line in test:
    # Check if the line starts with a year (assuming it starts a new entry)
    if re.match(r"\d{4}", line):
        if temp_lines:
            parsed_data = extract_data(temp_lines)
            if parsed_data:
                data.append(parsed_data)
            temp_lines = []

    # Append the current line to temp_lines
    temp_lines.append(line)

# Process the last accumulated lines
if temp_lines:
    parsed_data = extract_data(temp_lines)
    if parsed_data:
        data.append(parsed_data)

print(data)

[('2020', 'NISSAN  NAVARA M/T 2.5 EL 4X2 (D)  [ No battery ]', 'N/A', 'JAD5115', 'BROWN', '588,640', '647,504')]


In [72]:
locations = ['bacolod', 'cagayan_de_oro', 'cebu', 'dagupan', 'davao', 'guiguinto', 'iloilo', 'lipa', 'naga', 'san_fernando']
dfs = []
#Extract Data Function
def extract_data(lines):
    entry = ' '.join(lines)
    pattern = re.compile(
        r'''(\d{4})  # Year
        \s+(.*)  # Unit Description
        \s+(NO\sMILEAGE|unverified|-|N/A|(?:\d{1,3}(?:,\d{3})*)) # Kilometrage
        \s+([\d\s/]+|[\w/\d]+(?:\s*/\s*[\w/\d]+)?) # PN/CS
        \s+([\w\s/]+) # Color
        \s+((?:\d{1,3}(?:,\d{3})*)) # Reg Bidding Price
        \s+((?:\d{1,3}(?:,\d{3})*)) # Bid Now Price''',
        re.VERBOSE
    )
    match = pattern.search(entry)
    if match:
        return match.groups()
    return None

for loc in locations:

  #Read PDF File
  pdf_document = f'/content/drive/MyDrive/Webscraping Datasets (BPI)/{loc}.pdf'
  pdf = fitz.open(pdf_document)

  #Extract Text
  text = ""
  for page_num in range(len(pdf)):
      page = pdf.load_page(page_num)
      text += page.get_text()
  lines = text.split('\n')

  #Extract Data
  data = []
  temp_lines = []

  for line in lines:
      # Check if the line starts with a year (assuming it starts a new entry)
      if re.match(r"\d{4}", line):
          if temp_lines:
              parsed_data = extract_data(temp_lines)
              if parsed_data:
                  data.append(parsed_data)
              temp_lines = []

      # Append the current line to temp_lines
      temp_lines.append(line)

  # Process the last accumulated lines
  if temp_lines:
      parsed_data = extract_data(temp_lines)
      if (parsed_data):
          data.append(parsed_data)

  # Define the column names
  columns = ["year", "unit_description", "kilometrage", "pn/cs", "color", "reg_bidding_price", "bid_now_price"]

  # Create a DataFrame from the extracted data
  exec('df_{} = pd.DataFrame(data, columns=columns)'.format(loc))
  exec('dfs.append(df_{})'.format(loc))

In [68]:
df_cebu

,year,unit_description,kilometrage,pn/cs,color,reg_bidding_price,bid_now_price
0,2023,Chery Tiggo 7 Pro A/T 1.5L [ Lapu-Lapu ],"6,198",GAK9642,Red,"842,100","926,310"
1,2022,Chery Tiggo 7 A/T 1.5L Pro,"20,232",GAS5001,RED,"594,000","653,400"
2,2023,Geely Emgrand Premium A/T 1.5L,"3,186",GBA6598,GREY,"755,175","830,693"
3,2007,Kia Picanto M/T 1.6L,"139,132",YFP796,Red,"76,200","83,820"
4,2023,Kia Motors Stonic A/T 1.4L LX,"2,817",GBA2538,White,"660,800","726,880"
5,2022,MG ZS T A/T 1.3L [ Leyte Registered ],"7,530",HAG3740,Black,"675,000","742,500"
6,2022,MG MG5 M/T 1.5L [ Leyte registered ],"17,413",HAG1280,White,"361,200","397,320"
7,2022,MG MG5 M/T 1.5L,"12,583",GAR1626,BLACK,"338,325","372,158"
8,2021,MG ZS A/T 1.5L,"22,585",GAW1139,RED,"458,325","504,158"
9,2022,Mirage Mitsubishi A/T 1.2L GLX G,"13,221",GAW3053,TITANIUM GRAY METALLIC,"400,125","440,138"


In [ ]:
'''
bacolod = 21 /
cagayan_de_oro = 59 /
cebu = 54 X (missing the 2 rows with pn/cs 130107 - 52)
dagupan = 10 /
davao = 38 /
guiguinto = 34 /
iloilo = 37 /
lipa = 16 /
naga = 2 /
san_fernando = 17 X (missing the row with no kilometrage - 16)
'''

## Concatenate Dataframes

In [77]:
df_concat = pd.concat(dfs)
df_concat

,year,unit_description,kilometrage,pn/cs,color,reg_bidding_price,bid_now_price
0,2022,Changan CS35 A/T 1.4L Plus 7DCT Luxe [ No se...,"15,378",N1P916,White,"535,990","589,589"
1,2021,Chery Tiggo5x A/T 1.5L Comfort 5STR ...,"23,288",FAF2671,Blue,"337,350","371,085"
2,2021,Hyundai Accent M/T 1.4L GL [ LTO Pontevedra-...,"77,161",FAI8306,Silver,"407,170","447,887"
3,2020,Hyundai Accent A/T 1.4L GL [ Car's aircon is...,"33,291",FAI3373,Red,"479,340","527,274"
4,2021,Maxus G50 A/T 1.5L Elite Turbo DCT [ No tool...,"30,596",FAI5987,White,"538,980","592,878"
...,...,...,...,...,...,...,...
11,2021,Geely Okavango A/T 1500 Urban [ unencumbered...,"22,164",CAX3077,Black,"848,700","933,570"
12,2024,Mitsubishi L300 M/T 2268 FB EXCEED 2.2D w/ du...,"2,939",Y3S699,Polar White,"987,000","1,085,700"
13,2020,Mitsubishi L300 M/T 2268 FB (Head only) (D) ...,"110,993",CAV8858,White,"302,400","332,640"
14,2023,Toyota Wigo A/T 998 1.0G,"13,436",CBE5969,Yellow,"526,050","578,655"
